# Prepare Clinical data

In [21]:
LONG_RESP_THRESHOLD = 12
DROP_BOR = True

In [22]:
import pandas as pd
import numpy as np

source_map = {
    'doi:10.1016/j.cell.2015.07.061': 'Hugo et al.',
    'doi:10.1172/JCI78954DS1': 'Kwong et al.',
    'doi:10.1158/1078-0432.CCR-18-0720': 'Yan et al.',
    'doi:10.3390/cancers11081203': 'Louveau et al.',
    'doi:10.1038/ncomms6694': 'Long et al.',
    'doi:10.1158/1078-0432.CCR-13-3122': 'Rizos et al.',
    "doi:10.3390/cancers12082224": 'Blateau et al.',
    "doi:10.1200/PO.16.00054": 'Catalanotti et al.',
    'doi:10.1158/2159-8290.CD-13-0617': 'Van Allen at al.'
}

clinical = pd.read_csv("../dataset/original/clinical.csv")
clinical['patientID'] = clinical['patientID'].str.replace(r'^HL_', 'HL_Shi-', regex=True)
clinical['source'] = clinical['source'].replace(source_map)
clinical = clinical.drop(columns=['id', 'creation_datetime', 'original_patientID', 'OS_status', 'OS_month', 'CNA_data', 'SNV_data', 'GEX_data'])

if DROP_BOR == True:
    clinical = clinical.drop(columns=['BOR'])

print(clinical.shape)
clinical.head()

(417, 14)


,patientID,sex,age,AJCC_stage,M_stage,LDH,PFS_status,PFS_month,drug,BRAF_mut,brain_metastasis,immunotherapy_treatment,pre_MAPKi_treatment,source
0,BS_000,male,56,IV,NaN,normal,1.0,30.5,dabrafenib + trametinib,V600E,no,no,no,Blateau et al.
1,BS_001,male,86,IV,NaN,normal,0.0,24.1,dabrafenib,V600E,no,no,no,Blateau et al.
2,BS_002,female,47,IV,NaN,normal,0.0,14.1,dabrafenib + trametinib,V600E,no,no,no,Blateau et al.
3,BS_003,female,50,IV,NaN,NaN,1.0,1.6,vemurafenib,V600E,no,no,no,Blateau et al.
4,BS_004,female,47,IV,NaN,elevated,1.0,11.9,dabrafenib + trametinib,V600K,no,no,no,Blateau et al.


In [23]:
clinical['source'].unique()

array(['Blateau et al.', 'Catalanotti et al.', 'Van Allen at al.',
       'Yan et al.', 'Louveau et al.', 'Rizos et al.', 'Long et al.',
       'Kwong et al.', 'Hugo et al.'], dtype=object)

## Add Target label

In [24]:
# Check unexpected PFS_status values
print(clinical['PFS_status'].value_counts(dropna=False))

PFS_status
1.0    341
0.0     74
NaN      2
Name: count, dtype: int64


In [25]:
clinical['PFS_status'] = pd.to_numeric(clinical['PFS_status'], errors='coerce')

def label_pfs(row):
    if row['PFS_month'] >= LONG_RESP_THRESHOLD:
        return 2
    elif row['PFS_month'] < 6 and row['PFS_status'] == 1:
        return 0
    elif row['PFS_month'] >= 6 and row['PFS_month'] < LONG_RESP_THRESHOLD and row['PFS_status'] == 1:
        return 1
    return np.nan

clinical['pfs_label'] = clinical.apply(label_pfs, axis=1)
clinical = clinical.drop(columns=['PFS_status']).dropna(subset=['pfs_label'])
clinical['pfs_label'] = clinical['pfs_label'].astype(int)

print(clinical.shape)
clinical.head()


(386, 14)


,patientID,sex,age,AJCC_stage,M_stage,LDH,PFS_month,drug,BRAF_mut,brain_metastasis,immunotherapy_treatment,pre_MAPKi_treatment,source,pfs_label
0,BS_000,male,56,IV,NaN,normal,30.5,dabrafenib + trametinib,V600E,no,no,no,Blateau et al.,2
1,BS_001,male,86,IV,NaN,normal,24.1,dabrafenib,V600E,no,no,no,Blateau et al.,2
2,BS_002,female,47,IV,NaN,normal,14.1,dabrafenib + trametinib,V600E,no,no,no,Blateau et al.,2
3,BS_003,female,50,IV,NaN,NaN,1.6,vemurafenib,V600E,no,no,no,Blateau et al.,0
4,BS_004,female,47,IV,NaN,elevated,11.9,dabrafenib + trametinib,V600K,no,no,no,Blateau et al.,1


In [26]:
clinical['pfs_label'].value_counts().sort_index()

pfs_label
0    209
1     91
2     86
Name: count, dtype: int64

## OHE 'drug' and 'BRAF_mut'

In [27]:
drug_series = clinical['drug'].fillna('').str.replace(r'\s*\+\s*', '+', regex=True).str.strip()
drug_dummies = drug_series.str.get_dummies(sep='+').drop(columns=[''], errors='ignore')
clinical = pd.concat([clinical.drop('drug', axis=1), drug_dummies], axis=1)

braf_series = clinical['BRAF_mut'].fillna('').str.replace(r'\s*;\s*', ';', regex=True).str.strip()
braf_mut_dummies = braf_series.str.get_dummies(sep=';').drop(columns=[''], errors='ignore')
clinical = pd.concat([clinical.drop('BRAF_mut', axis=1), braf_mut_dummies], axis=1)
clinical = clinical.drop(columns=['nan'], errors='ignore')

print(clinical.shape)
clinical.head()

(386, 22)


,patientID,sex,age,AJCC_stage,M_stage,LDH,PFS_month,brain_metastasis,immunotherapy_treatment,pre_MAPKi_treatment,...,cobimetinib,dabrafenib,trametinib,vemurafenib,C195W,K601I,MND,V600E,V600K,V600R
0,BS_000,male,56,IV,NaN,normal,30.5,no,no,no,...,0,1,1,0,0,0,0,1,0,0
1,BS_001,male,86,IV,NaN,normal,24.1,no,no,no,...,0,1,0,0,0,0,0,1,0,0
2,BS_002,female,47,IV,NaN,normal,14.1,no,no,no,...,0,1,1,0,0,0,0,1,0,0
3,BS_003,female,50,IV,NaN,NaN,1.6,no,no,no,...,0,0,0,1,0,0,0,1,0,0
4,BS_004,female,47,IV,NaN,elevated,11.9,no,no,no,...,0,1,1,0,0,0,0,0,1,0


In [28]:
clinical.to_csv(f"../dataset/created/clinical.csv", index=False)